In [4]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from pydantic import BaseModel, Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Literal, TypedDict, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display, Markdown
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

class llm_schema(BaseModel):
    tasks: List[str] = Field(..., description="A list of tasks to be performed by the worker.")

llm_with_schema = llm_openai.with_structured_output(llm_schema)


class graph_schema(TypedDict):
    tasks: List[str]
    query: str
    results: List[str]
    summary: str

def orchestrator_node(state: graph_schema) -> graph_schema:
    # Fetching the user query from the state
    user_query = state['query']
    # Create the prompt for the LLM
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an orchestrator that breaks down a user query into tasks for the worker."),
            ("user", f"User query: {user_query}. Please generate one prompt per task for the worker to complete. Return the tasks in a list format."),
        ]
    )
    # Create the chain
    chain = prompt | llm_with_schema
    # Run the chain with the user query as input
    response = chain.invoke({"query": user_query})
    # Update the state with the generated tasks
    state['tasks'] = response.tasks
    return state

def execute(query:str) :
    response = llm_openai.invoke(f"Please execute this task {query}")
    return response.content

from concurrent.futures import ThreadPoolExecutor

def worker_node(state: graph_schema) -> graph_schema:

    tasks = state['tasks']
    results = []
    with ThreadPoolExecutor(max_workers=len(tasks)) as executor:

        results_futures = executor.map(execute, tasks)
        for result in results_futures:
            results.append(result)

    state['results'] = results

    return state

def collector_node(state: graph_schema):
    results = state['results']

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a collector that summarizes the results from the worker."),
            ("user", f"Here are the results from the worker: {results}. Please summarize these results in a concise manner."),
        ]
    )

    chain = prompt | llm_openai

    summary = chain.invoke({"results": results})

    state['summary'] = summary.content

    return state




graph=StateGraph(graph_schema)

graph.add_node("orchestrator_node",orchestrator_node)
graph.add_node("worker_node",worker_node)
graph.add_node("collector_node",collector_node)

graph.add_edge(START, "orchestrator_node")
graph.add_edge("orchestrator_node", "worker_node")
graph.add_edge("worker_node", "collector_node")
graph.add_edge("collector_node", END)


f_graph=graph.compile()

# Display Mermaid text instead of trying to render as PNG
print(f_graph.get_graph().draw_mermaid())

result=f_graph.invoke(
    {
        "query": "What is the capital of France and what is the population of Paris? & What is the capital of Germany and what is the population of Berlin?",
        "tasks": [],
        "results": [],
        "summary": "",
    }
)

print(result['summary'])

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	orchestrator_node(orchestrator_node)
	worker_node(worker_node)
	collector_node(collector_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> orchestrator_node;
	orchestrator_node --> worker_node;
	worker_node --> collector_node;
	collector_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=llm_schema(tasks=['Provid...on figure for Berlin.']), input_type=llm_schema])
  return self.__pydantic_serializer__.to_python(


**Summary of the worker’s findings**

- **France**  
  - Capital: **Paris**  
  - Latest population (city proper, 2023): **≈ 2.17 million** (2,165,423 inhabitants)

- **Germany**  
  - Capital: **Berlin**  
  - Latest population (2023‑2024 estimate): **≈ 3.85 million** residents
